# 00 Setup

Se ejecuta una vez, a mano, antes del job. Crea el volume, las carpetas, las tablas
y los permisos del grupo `mine-group8`. Carga el World Port Index en `oceanwatch_puertos`.

Catálogo `dbr_acad_ingsistemas_mine`, esquema `default`. En el cluster los archivos van con Python sobre `/Volumes`.


In [ ]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "dbr_acad_ingsistemas_mine"
SCHEMA = "default"
VOLUME = "oceanwatch_data"

VOLUME_NAME = f"`{CATALOG}`.`{SCHEMA}`.`{VOLUME}`"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_ZIPS = f"{VOLUME_ROOT}/raw_zips"
LANDING = f"{VOLUME_ROOT}/landing"
CHECKPOINTS = f"{VOLUME_ROOT}/checkpoints"
SCHEMAS = f"{VOLUME_ROOT}/_schemas"
REF = f"{VOLUME_ROOT}/ref"
STAGING = f"{RAW_ZIPS}/_staging"

BRONZE = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_bronze_ais`"
SILVER = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_silver_ais`"
CUARENTENA = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_cuarentena`"
METRICAS = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_metricas`"
PUERTOS = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_puertos`"
GOLD_TRAY = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_trayectorias`"
GOLD_PUERTO = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_presencia_puertos`"
GOLD_TRAFICO = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_gold_trafico_celda_hora`"

import os

print(VOLUME_ROOT)


## Volume y carpetas

`raw_zips/` guarda el zip, `landing/` el CSV que lee bronze, `checkpoints/` los
checkpoints de los streams, `ref/` el World Port Index y `_schemas/` el esquema de Auto Loader.


In [ ]:
spark.sql(f"""
CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
COMMENT 'Archivos de OceanWatch: zips NOAA, CSV de landing, checkpoints y referencias.'
""")

for ruta in (RAW_ZIPS, LANDING, CHECKPOINTS, SCHEMAS, REF):
    os.makedirs(ruta, exist_ok=True)
    print("Listo", ruta)


## Tablas

Bronze guarda los datos crudos, una fila por `(MMSI, BaseDateTime)`. Silver guarda
las filas que pasan las reglas de la Parte 1. Cuarentena guarda las que no, con `motivo`.
`oceanwatch_metricas` anota cada ejecución de bronze y cada batch de silver. Las tres tablas gold se rehacen por `fecha`.


In [ ]:
# NOTE: just add the name of the table to the dictionary, not {nombre}
tablas = {
    BRONZE: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING COMMENT 'Ruta del CSV en landing.',
          _ingested_at TIMESTAMP COMMENT 'Instante en que el batch escribio la fila.'
        )
        USING DELTA
        COMMENT 'Bronze AIS: mensaje crudo. Una fila por (MMSI, BaseDateTime).'
    """,
    SILVER: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          fecha DATE
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Silver AIS: filas que pasan MMSI, timestamp, latitud, longitud, SOG y COG.'
    """,
    CUARENTENA: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          MMSI STRING,
          BaseDateTime TIMESTAMP,
          LAT DOUBLE,
          LON DOUBLE,
          SOG DOUBLE,
          COG DOUBLE,
          Heading DOUBLE,
          VesselName STRING,
          IMO STRING,
          CallSign STRING,
          VesselType INT,
          Status INT,
          Length DOUBLE,
          Width DOUBLE,
          Draft DOUBLE,
          Cargo STRING,
          TransceiverClass STRING,
          _source_file STRING,
          _ingested_at TIMESTAMP,
          motivo STRING COMMENT 'Reglas fallidas, separadas por coma.',
          _clave STRING COMMENT 'Hash de la fila. El MERGE no la reinserta.'
        )
        USING DELTA
        COMMENT 'Filas AIS que no entran a silver, con el motivo.'
    """,
    METRICAS: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          fecha DATE COMMENT 'Dia del parametro del job.',
          etapa STRING COMMENT 'bronze o silver.',
          filas_antes BIGINT COMMENT 'Conteo de la tabla bronze antes del stream.',
          filas_despues BIGINT COMMENT 'Conteo de la tabla bronze despues del stream.',
          filas_entrada BIGINT COMMENT 'Filas del batch de silver.',
          filas_validas BIGINT COMMENT 'Filas del batch que pasan las reglas.',
          filas_cuarentena BIGINT COMMENT 'Filas del batch que van a cuarentena.',
          ejecutado_en TIMESTAMP
        )
        USING DELTA
        COMMENT 'Conteos de cada ejecucion de bronze y de cada batch de silver.'
    """,
    PUERTOS: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          numero INT COMMENT 'World Port Index Number.',
          nombre STRING,
          pais STRING,
          latitud DOUBLE,
          longitud DOUBLE
        )
        USING DELTA
        COMMENT 'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'
    """,
    GOLD_TRAY: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          fecha DATE,
          MMSI STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP,
          lat_inicio DOUBLE,
          lon_inicio DOUBLE,
          lat_fin DOUBLE,
          lon_fin DOUBLE,
          distancia_km DOUBLE COMMENT 'Suma de tramos haversine del dia, radio 6371 km.',
          sog_media DOUBLE
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Una trayectoria por buque y dia, calculada sobre silver.'
    """,
    GOLD_PUERTO: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          fecha DATE,
          MMSI STRING,
          puerto STRING,
          pais STRING,
          n_posiciones BIGINT,
          inicio TIMESTAMP,
          fin TIMESTAMP
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Buque con al menos una posicion del dia junto a un puerto del WPI.'
    """,
    GOLD_TRAFICO: """
        CREATE TABLE IF NOT EXISTS {nombre} (
          fecha DATE,
          hora INT COMMENT 'Hora UTC de BaseDateTime.',
          h3_cell STRING COMMENT 'Celda H3 resolucion 8, la misma de la Parte 1.',
          n_posiciones BIGINT,
          n_buques BIGINT
        )
        USING DELTA
        PARTITIONED BY (fecha)
        COMMENT 'Matriz de trafico celda H3 por hora, un dia.'
    """,
}

for nombre, plantilla in tablas.items():
    spark.sql(plantilla.format(nombre=nombre))
    print("Tabla", nombre)


## World Port Index

La Parte 1 leyó `UpdatedPub150.csv` (columnas `Main Port Name`, `Country Code`,
`Latitude`, `Longitude`). Si el archivo ya está en `ref/` o en el volume de la Parte 1,
se copia y se carga. Si no está, hay que subirlo a `ref/UpdatedPub150.csv` y volver a
ejecutar esta celda.


In [ ]:
# NOTE: fix this. we don't have access to the volume of Parte 1
from pathlib import Path
import shutil
from pyspark.sql import functions as F

destino = Path(REF) / "UpdatedPub150.csv"
candidatos = [
    destino,
    Path("/Volumes/proyecto_datos/default/data/UpdatedPub150.csv"),
]
origen = next((ruta for ruta in candidatos if ruta.is_file()), None)
if origen is None:
    raise FileNotFoundError(
        "No esta UpdatedPub150.csv. Subelo a "
        f"{destino} y vuelve a ejecutar esta celda."
    )
if origen.resolve() != destino.resolve():
    shutil.copyfile(origen, destino)
    print("Copiado", origen, "->", destino)

(
    spark.read.option("header", True).csv(str(destino))
    .select(
        F.col("World Port Index Number").cast("int").alias("numero"),
        F.col("Main Port Name").alias("nombre"),
        F.col("Country Code").alias("pais"),
        F.col("Latitude").cast("double").alias("latitud"),
        F.col("Longitude").cast("double").alias("longitud"),
    )
    .where(F.col("latitud").isNotNull() & F.col("longitud").isNotNull())
    .write.mode("overwrite")
    .saveAsTable(PUERTOS)
)
spark.sql(f"COMMENT ON TABLE {PUERTOS} IS 'World Port Index (UpdatedPub150), el mismo catalogo de la Parte 1.'")
print("Puertos", spark.table(PUERTOS).count())


## Permisos

El grupo `mine-group8` puede leer y escribir el volume, y leer y modificar las tablas.


In [ ]:
principal = "`mine-group8`"
spark.sql(f"GRANT READ VOLUME ON VOLUME {VOLUME_NAME} TO {principal}")
spark.sql(f"GRANT WRITE VOLUME ON VOLUME {VOLUME_NAME} TO {principal}")
for tabla in (BRONZE, SILVER, CUARENTENA, METRICAS, PUERTOS, GOLD_TRAY, GOLD_PUERTO, GOLD_TRAFICO):
    spark.sql(f"GRANT SELECT ON TABLE {tabla} TO {principal}")
    spark.sql(f"GRANT MODIFY ON TABLE {tabla} TO {principal}")
    print("GRANT", tabla)
